# Valencia Airbnb Price Prediction
## Machine Learning Course Project

**Objective**: Predict the price of Airbnb apartments in Valencia, Spain using multiple machine learning techniques.

**Methods**:
1. Regularized Generalized Linear Models (Ridge, Lasso, ElasticNet)
2. Regression and Classification Trees
3. Ensemble Methods: Bagging, Random Forests
4. Ensemble Methods: Boosting (XGBoost, LightGBM with GPU)
5. Deep Learning: Neural Networks (PyTorch with CUDA)

**Author**: ML Course Student
**Date**: 2025-11-16

## 1. Setup and Data Acquisition

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# For reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

In [ ]:
# Check CUDA availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

### 1.1 Data Acquisition

We'll fetch Airbnb data for Valencia from Inside Airbnb (http://insideairbnb.com/), a public dataset that provides comprehensive information about Airbnb listings.

In [ ]:
import osimport requestsfrom pathlib import Path# Create data directorydata_dir = Path('../data')data_dir.mkdir(exist_ok=True)# URLs for Valencia Airbnb data from Inside Airbnbvalencia_url = "http://data.insideairbnb.com/spain/valencia/valencia/2024-09-18/data/listings.csv.gz"listings_file = data_dir / 'valencia_listings.csv.gz'# Download if not existsif not listings_file.exists():    print("Downloading Valencia Airbnb listings data...")    # Use requests with headers to avoid 403    headers = {        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'    }    response = requests.get(valencia_url, headers=headers, stream=True)    response.raise_for_status()        with open(listings_file, 'wb') as f:        for chunk in response.iter_content(chunk_size=8192):            f.write(chunk)    print(f"Data downloaded successfully to {listings_file}")else:    print(f"Data already exists at {listings_file}")# Load the dataprint("\nLoading data...")df_raw = pd.read_csv(listings_file, compression='gzip')print(f"Data loaded successfully! Shape: {df_raw.shape}")print(f"Number of listings: {len(df_raw):,}")print(f"Number of features: {df_raw.shape[1]}")

## 2. Application Context

### Business Context
Valencia is a major tourist destination in Spain, attracting millions of visitors annually. Understanding Airbnb pricing dynamics is valuable for:

- **Hosts**: Setting competitive prices for their properties
- **Guests**: Finding fair-priced accommodations
- **Investors**: Identifying profitable investment opportunities
- **Policy Makers**: Understanding the short-term rental market

### Dataset Description
The Inside Airbnb dataset contains detailed information about:
- Property characteristics (type, size, amenities)
- Location (neighborhood, coordinates)
- Host information (response time, ratings)
- Pricing and availability
- Guest reviews and ratings

In [ ]:
# Display basic information
print("Dataset Overview:")
print("=" * 80)
df_raw.info()

In [ ]:
# Display first few rows
df_raw.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Create a working copy
df = df_raw.copy()

# Convert price to numeric (remove $ and commas)
df['price_numeric'] = df['price'].replace('[\$,]', '', regex=True).astype(float)

# Basic statistics
print("Price Statistics:")
print("=" * 80)
print(df['price_numeric'].describe())
print(f"\nMissing values: {df['price_numeric'].isna().sum()}")

### 3.1 Target Variable Distribution

In [ ]:
# Remove extreme outliers for better visualization
price_clean = df[df['price_numeric'] > 0]['price_numeric']
price_q99 = price_clean.quantile(0.99)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Price Distribution', 'Price Distribution (Log Scale)']
)

# Histogram
fig.add_trace(
    go.Histogram(x=price_clean[price_clean <= price_q99], nbinsx=50, name='Price'),
    row=1, col=1
)

# Log scale histogram
fig.add_trace(
    go.Histogram(x=np.log1p(price_clean), nbinsx=50, name='Log(Price)'),
    row=1, col=2
)

fig.update_layout(height=400, showlegend=False, title_text="Price Analysis")
fig.show()

print(f"Skewness (original): {price_clean.skew():.2f}")
print(f"Skewness (log): {np.log1p(price_clean).skew():.2f}")

### 3.2 Property Types and Room Types

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'bar'}, {'type': 'bar'}]],
    subplot_titles=['Distribution by Room Type', 'Average Price by Room Type']
)

# Count by room type
room_counts = df['room_type'].value_counts()
fig.add_trace(
    go.Bar(x=room_counts.index, y=room_counts.values, name='Count'),
    row=1, col=1
)

# Average price by room type
room_price = df.groupby('room_type')['price_numeric'].mean().sort_values(descending=True)
fig.add_trace(
    go.Bar(x=room_price.index, y=room_price.values, name='Avg Price', marker_color='orange'),
    row=1, col=2
)

fig.update_layout(height=400, showlegend=False)
fig.show()

### 3.3 Geographic Distribution

In [ ]:
# Map of listings
df_map = df[df['price_numeric'] > 0].copy()
df_map = df_map[df_map['price_numeric'] <= df_map['price_numeric'].quantile(0.99)]

fig = px.scatter_mapbox(
    df_map,
    lat="latitude",
    lon="longitude",
    color="price_numeric",
    size="price_numeric",
    hover_data=["name", "room_type", "neighbourhood_cleansed"],
    color_continuous_scale="Viridis",
    zoom=11,
    height=600,
    title="Airbnb Listings in Valencia (colored by price)"
)

fig.update_layout(mapbox_style="open-street-map")
fig.show()

### 3.4 Correlation Analysis

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['price_numeric', 'accommodates', 'bedrooms', 'beds', 'bathrooms_text',
                'minimum_nights', 'maximum_nights', 'number_of_reviews', 
                'review_scores_rating', 'review_scores_cleanliness', 'review_scores_checkin',
                'review_scores_communication', 'review_scores_location', 'review_scores_value',
                'calculated_host_listings_count', 'availability_365']

# Convert bathrooms_text to numeric
df['bathrooms_numeric'] = df['bathrooms_text'].str.extract('(\d+\.?\d*)').astype(float)

# Create correlation matrix
corr_cols = ['price_numeric', 'accommodates', 'bedrooms', 'beds', 'bathrooms_numeric',
             'number_of_reviews', 'review_scores_rating', 'availability_365']

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Key Features', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

## 4. Data Pre-processing and Feature Engineering

### 4.1 Feature Selection and Cleaning

In [ ]:
print("Starting feature engineering...")
print("=" * 80)

# Create a new dataframe for modeling
df_model = df.copy()

# Target variable
df_model = df_model[df_model['price_numeric'] > 0].copy()

# Remove extreme outliers (above 99th percentile)
price_threshold = df_model['price_numeric'].quantile(0.99)
df_model = df_model[df_model['price_numeric'] <= price_threshold].copy()

print(f"Removed {len(df) - len(df_model)} outliers")
print(f"Remaining samples: {len(df_model):,}")

In [ ]:
# Feature engineering
from sklearn.preprocessing import LabelEncoder

# Numeric features
df_model['accommodates'] = df_model['accommodates'].fillna(df_model['accommodates'].median())
df_model['bedrooms'] = df_model['bedrooms'].fillna(df_model['bedrooms'].median())
df_model['beds'] = df_model['beds'].fillna(df_model['beds'].median())
df_model['bathrooms'] = df_model['bathrooms_numeric'].fillna(df_model['bathrooms_numeric'].median())

# Review scores
review_cols = ['review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
               'review_scores_checkin', 'review_scores_communication', 'review_scores_location',
               'review_scores_value']

for col in review_cols:
    if col in df_model.columns:
        df_model[col] = df_model[col].fillna(df_model[col].median())

# Categorical features
df_model['room_type_encoded'] = LabelEncoder().fit_transform(df_model['room_type'].fillna('Unknown'))
df_model['property_type_encoded'] = LabelEncoder().fit_transform(
    df_model['property_type'].fillna('Unknown').astype(str)
)

# Boolean features
df_model['instant_bookable'] = (df_model['instant_bookable'] == 't').astype(int)
df_model['host_is_superhost'] = (df_model['host_is_superhost'] == 't').astype(int)
df_model['host_identity_verified'] = (df_model['host_identity_verified'] == 't').astype(int)

# Availability
df_model['availability_365'] = df_model['availability_365'].fillna(0)

# Location features
df_model['latitude'] = df_model['latitude'].fillna(df_model['latitude'].median())
df_model['longitude'] = df_model['longitude'].fillna(df_model['longitude'].median())

# Distance from city center (Valencia center: 39.4699, -0.3763)
valencia_center_lat, valencia_center_lon = 39.4699, -0.3763
df_model['distance_from_center'] = np.sqrt(
    (df_model['latitude'] - valencia_center_lat)**2 + 
    (df_model['longitude'] - valencia_center_lon)**2
)

# Text features - amenities count
df_model['amenities_count'] = df_model['amenities'].fillna('[]').apply(
    lambda x: len(eval(x)) if x != '[]' else 0
)

# Host characteristics
df_model['host_response_rate'] = df_model['host_response_rate'].replace('%', '', regex=True).astype(float) / 100
df_model['host_response_rate'] = df_model['host_response_rate'].fillna(df_model['host_response_rate'].median())

df_model['host_acceptance_rate'] = df_model['host_acceptance_rate'].replace('%', '', regex=True).astype(float) / 100
df_model['host_acceptance_rate'] = df_model['host_acceptance_rate'].fillna(df_model['host_acceptance_rate'].median())

print("Feature engineering completed!")

### 4.2 Feature Selection

In [ ]:
# Select features for modeling
feature_columns = [
    # Property characteristics
    'accommodates', 'bedrooms', 'beds', 'bathrooms',
    
    # Location
    'latitude', 'longitude', 'distance_from_center',
    
    # Property type
    'room_type_encoded', 'property_type_encoded',
    
    # Amenities
    'amenities_count',
    
    # Reviews
    'number_of_reviews', 'review_scores_rating', 'review_scores_accuracy',
    'review_scores_cleanliness', 'review_scores_checkin', 
    'review_scores_communication', 'review_scores_location', 'review_scores_value',
    
    # Host
    'host_is_superhost', 'host_identity_verified',
    'host_response_rate', 'host_acceptance_rate',
    
    # Booking
    'instant_bookable', 'minimum_nights', 'availability_365'
]

# Create X and y
X = df_model[feature_columns].copy()
y = df_model['price_numeric'].copy()

# Check for any remaining NaN values
print("Missing values per feature:")
print(X.isna().sum().sort_values(ascending=False).head(10))

# Fill any remaining NaN values with median
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

print(f"\nFinal dataset shape: {X_imputed.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures: {list(X_imputed.columns)}")

### 4.3 Train-Test Split and Scaling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]:,}")
print(f"Test set size: {X_test.shape[0]:,}")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nData preprocessing completed!")
print("=" * 80)

## 5. Model Training and Evaluation

We'll train multiple models with different complexities and compare their:
- **Training time** (showing GPU acceleration where applicable)
- **Prediction accuracy** (R², RMSE, MAE)
- **Model complexity** (number of parameters/trees)

### Performance Metrics

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time

# Dictionary to store results
results = []

def evaluate_model(name, model, X_train, X_test, y_train, y_test, train_time, complexity="N/A"):
    """Evaluate model and store results"""
    # Predictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Metrics
    metrics = {
        'Model': name,
        'Complexity': complexity,
        'Train Time (s)': round(train_time, 4),
        'Train R²': round(r2_score(y_train, y_pred_train), 4),
        'Test R²': round(r2_score(y_test, y_pred_test), 4),
        'Train RMSE': round(np.sqrt(mean_squared_error(y_train, y_pred_train)), 2),
        'Test RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred_test)), 2),
        'Train MAE': round(mean_absolute_error(y_train, y_pred_train), 2),
        'Test MAE': round(mean_absolute_error(y_test, y_pred_test), 2)
    }
    
    results.append(metrics)
    
    print(f"\n{name} Results:")
    print(f"  Complexity: {complexity}")
    print(f"  Training Time: {train_time:.4f}s")
    print(f"  Test R²: {metrics['Test R²']:.4f}")
    print(f"  Test RMSE: €{metrics['Test RMSE']:.2f}")
    print(f"  Test MAE: €{metrics['Test MAE']:.2f}")
    
    return metrics

print("Evaluation functions ready!")

### 5.1 Regularized Generalized Linear Models

Starting with different regularization techniques:
- **Ridge**: L2 regularization
- **Lasso**: L1 regularization (feature selection)
- **ElasticNet**: Combined L1 + L2 regularization

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV

print("=" * 80)
print("REGULARIZED LINEAR MODELS")
print("=" * 80)

# Ridge Regression with hyperparameter tuning
print("\n1. Ridge Regression (L2 Regularization)...")
start_time = time.time()

ridge_params = {'alpha': [0.1, 1.0, 10.0, 100.0, 1000.0]}
ridge = GridSearchCV(Ridge(), ridge_params, cv=5, scoring='r2', n_jobs=-1)
ridge.fit(X_train_scaled, y_train)

ridge_time = time.time() - start_time
print(f"Best alpha: {ridge.best_params_['alpha']}")
evaluate_model('Ridge', ridge.best_estimator_, X_train_scaled, X_test_scaled, 
               y_train, y_test, ridge_time, f"α={ridge.best_params_['alpha']}")

In [ ]:
# Lasso Regression with hyperparameter tuning
print("\n2. Lasso Regression (L1 Regularization)...")
start_time = time.time()

lasso_params = {'alpha': [0.1, 1.0, 10.0, 100.0]}
lasso = GridSearchCV(Lasso(max_iter=5000), lasso_params, cv=5, scoring='r2', n_jobs=-1)
lasso.fit(X_train_scaled, y_train)

lasso_time = time.time() - start_time
print(f"Best alpha: {lasso.best_params_['alpha']}")

# Count non-zero coefficients (feature selection)
n_features_selected = np.sum(lasso.best_estimator_.coef_ != 0)
print(f"Features selected: {n_features_selected}/{len(X_train.columns)}")

evaluate_model('Lasso', lasso.best_estimator_, X_train_scaled, X_test_scaled, 
               y_train, y_test, lasso_time, f"α={lasso.best_params_['alpha']}, {n_features_selected} features")

In [ ]:
# ElasticNet Regression with hyperparameter tuning
print("\n3. ElasticNet (L1 + L2 Regularization)...")
start_time = time.time()

elastic_params = {
    'alpha': [0.1, 1.0, 10.0],
    'l1_ratio': [0.2, 0.5, 0.8]
}
elastic = GridSearchCV(ElasticNet(max_iter=5000), elastic_params, cv=5, scoring='r2', n_jobs=-1)
elastic.fit(X_train_scaled, y_train)

elastic_time = time.time() - start_time
print(f"Best params: alpha={elastic.best_params_['alpha']}, l1_ratio={elastic.best_params_['l1_ratio']}")

evaluate_model('ElasticNet', elastic.best_estimator_, X_train_scaled, X_test_scaled, 
               y_train, y_test, elastic_time, 
               f"α={elastic.best_params_['alpha']}, ratio={elastic.best_params_['l1_ratio']}")

### 5.2 Decision Tree Regression

In [ ]:
from sklearn.tree import DecisionTreeRegressor

print("\n" + "=" * 80)
print("TREE-BASED MODELS")
print("=" * 80)

# Decision Tree with different complexities
print("\n4. Decision Tree Regressor...")

tree_params = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

start_time = time.time()
tree = GridSearchCV(DecisionTreeRegressor(random_state=42), tree_params, 
                    cv=5, scoring='r2', n_jobs=-1)
tree.fit(X_train, y_train)
tree_time = time.time() - start_time

print(f"Best params: {tree.best_params_}")
evaluate_model('Decision Tree', tree.best_estimator_, X_train, X_test, 
               y_train, y_test, tree_time, 
               f"depth={tree.best_params_['max_depth']}, leaves~{tree.best_estimator_.get_n_leaves()}")

### 5.3 Ensemble Methods - Bagging and Random Forest

In [ ]:
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor

print("\n" + "=" * 80)
print("ENSEMBLE METHODS: BAGGING AND RANDOM FOREST")
print("=" * 80)

# Bagging Regressor
print("\n5. Bagging Regressor...")
start_time = time.time()

bagging_params = {
    'n_estimators': [50, 100, 200],
    'max_samples': [0.5, 0.7, 1.0]
}
bagging = GridSearchCV(
    BaggingRegressor(random_state=42, n_jobs=-1),
    bagging_params, cv=3, scoring='r2', n_jobs=2
)
bagging.fit(X_train, y_train)
bagging_time = time.time() - start_time

print(f"Best params: {bagging.best_params_}")
evaluate_model('Bagging', bagging.best_estimator_, X_train, X_test, 
               y_train, y_test, bagging_time, 
               f"n_estimators={bagging.best_params_['n_estimators']}")

In [ ]:
# Random Forest - different complexities
print("\n6. Random Forest (Small - Fast)...")
start_time = time.time()

rf_small = RandomForestRegressor(
    n_estimators=50, max_depth=10, random_state=42, n_jobs=-1
)
rf_small.fit(X_train, y_train)
rf_small_time = time.time() - start_time

evaluate_model('Random Forest (Small)', rf_small, X_train, X_test, 
               y_train, y_test, rf_small_time, "50 trees, depth=10")

In [ ]:
print("\n7. Random Forest (Medium - Moderate)...")
start_time = time.time()

rf_medium = RandomForestRegressor(
    n_estimators=100, max_depth=20, random_state=42, n_jobs=-1
)
rf_medium.fit(X_train, y_train)
rf_medium_time = time.time() - start_time

evaluate_model('Random Forest (Medium)', rf_medium, X_train, X_test, 
               y_train, y_test, rf_medium_time, "100 trees, depth=20")

In [ ]:
print("\n8. Random Forest (Large - Slow but Accurate)...")
start_time = time.time()

rf_large = RandomForestRegressor(
    n_estimators=200, max_depth=None, min_samples_split=2, 
    random_state=42, n_jobs=-1
)
rf_large.fit(X_train, y_train)
rf_large_time = time.time() - start_time

evaluate_model('Random Forest (Large)', rf_large, X_train, X_test, 
               y_train, y_test, rf_large_time, "200 trees, no depth limit")

### 5.4 Gradient Boosting Methods (GPU-Accelerated)

Using XGBoost and LightGBM with CUDA support for faster training.

In [ ]:
import xgboost as xgb
import lightgbm as lgb

print("\n" + "=" * 80)
print("GRADIENT BOOSTING METHODS (GPU-ACCELERATED)")
print("=" * 80)

# Check GPU availability for XGBoost and LightGBM
print(f"\nXGBoost version: {xgb.__version__}")
print(f"LightGBM version: {lgb.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# XGBoost with GPU (if available)
print("\n9. XGBoost (CPU - Small)...")
start_time = time.time()

xgb_small = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)
xgb_small.fit(X_train, y_train)
xgb_small_time = time.time() - start_time

evaluate_model('XGBoost (Small)', xgb_small, X_train, X_test, 
               y_train, y_test, xgb_small_time, "100 trees, depth=5")

In [ ]:
# XGBoost GPU-accelerated (if CUDA available)
if torch.cuda.is_available():
    print("\n10. XGBoost (GPU - Large)...")
    start_time = time.time()
    
    xgb_gpu = xgb.XGBRegressor(
        n_estimators=500,
        max_depth=10,
        learning_rate=0.05,
        tree_method='hist',
        device='cuda',
        random_state=42
    )
    xgb_gpu.fit(X_train, y_train)
    xgb_gpu_time = time.time() - start_time
    
    evaluate_model('XGBoost (GPU)', xgb_gpu, X_train, X_test, 
                   y_train, y_test, xgb_gpu_time, "500 trees, depth=10, GPU")
else:
    print("\n10. XGBoost (CPU - Large) [GPU not available]...")
    start_time = time.time()
    
    xgb_large = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )
    xgb_large.fit(X_train, y_train)
    xgb_large_time = time.time() - start_time
    
    evaluate_model('XGBoost (Large)', xgb_large, X_train, X_test, 
                   y_train, y_test, xgb_large_time, "300 trees, depth=8")

In [ ]:
# LightGBM with GPU (if available)
if torch.cuda.is_available():
    print("\n11. LightGBM (GPU)...")
    start_time = time.time()
    
    lgb_gpu = lgb.LGBMRegressor(
        n_estimators=500,
        max_depth=10,
        learning_rate=0.05,
        device='gpu',
        random_state=42,
        verbose=-1
    )
    lgb_gpu.fit(X_train, y_train)
    lgb_gpu_time = time.time() - start_time
    
    evaluate_model('LightGBM (GPU)', lgb_gpu, X_train, X_test, 
                   y_train, y_test, lgb_gpu_time, "500 trees, depth=10, GPU")
else:
    print("\n11. LightGBM (CPU)...")
    start_time = time.time()
    
    lgb_model = lgb.LGBMRegressor(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    lgb_model.fit(X_train, y_train)
    lgb_time = time.time() - start_time
    
    evaluate_model('LightGBM', lgb_model, X_train, X_test, 
                   y_train, y_test, lgb_time, "300 trees, depth=8")

### 5.5 Deep Learning - Neural Network (PyTorch with CUDA)

Building neural networks with different architectures to show training time differences.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

print("\n" + "=" * 80)
print("DEEP LEARNING - NEURAL NETWORKS (PyTorch)")
print("=" * 80)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

# Prepare data for PyTorch
X_train_tensor = torch.FloatTensor(X_train_scaled).to(device)
y_train_tensor = torch.FloatTensor(y_train.values).to(device)
X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
y_test_tensor = torch.FloatTensor(y_test.values).to(device)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

In [ ]:
# Neural Network class
class PricePredictor(nn.Module):
    def __init__(self, input_size, hidden_sizes, dropout=0.2):
        super(PricePredictor, self).__init__()
        layers = []
        
        # Input layer
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_size),
                nn.Dropout(dropout)
            ])
            prev_size = hidden_size
        
        # Output layer
        layers.append(nn.Linear(prev_size, 1))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x).squeeze()

def train_neural_network(model, train_loader, X_test, y_test, epochs=50, lr=0.001):
    """Train neural network and return training time"""
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
    
    train_time = time.time() - start_time
    return train_time

def evaluate_neural_network(model, X_train, X_test, y_train, y_test):
    """Evaluate neural network"""
    model.eval()
    with torch.no_grad():
        y_pred_train = model(X_train).cpu().numpy()
        y_pred_test = model(X_test).cpu().numpy()
    
    return y_pred_train, y_pred_test

print("Neural network utilities defined!")

In [ ]:
# Small Neural Network (Fast)
print("\n12. Neural Network (Small - Fast)...")
input_size = X_train_scaled.shape[1]

nn_small = PricePredictor(input_size, hidden_sizes=[32, 16], dropout=0.2).to(device)
nn_small_time = train_neural_network(nn_small, train_loader, X_test_tensor, y_test_tensor, 
                                      epochs=30, lr=0.001)

y_pred_train_small, y_pred_test_small = evaluate_neural_network(
    nn_small, X_train_tensor, X_test_tensor, y_train_tensor, y_test_tensor
)

# Calculate metrics manually for neural network
nn_small_metrics = {
    'Model': 'Neural Network (Small)',
    'Complexity': '2 layers (32→16), 30 epochs',
    'Train Time (s)': round(nn_small_time, 4),
    'Train R²': round(r2_score(y_train, y_pred_train_small), 4),
    'Test R²': round(r2_score(y_test, y_pred_test_small), 4),
    'Train RMSE': round(np.sqrt(mean_squared_error(y_train, y_pred_train_small)), 2),
    'Test RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred_test_small)), 2),
    'Train MAE': round(mean_absolute_error(y_train, y_pred_train_small), 2),
    'Test MAE': round(mean_absolute_error(y_test, y_pred_test_small), 2)
}
results.append(nn_small_metrics)

print(f"  Training Time: {nn_small_time:.4f}s ({device})")
print(f"  Test R²: {nn_small_metrics['Test R²']:.4f}")
print(f"  Test RMSE: €{nn_small_metrics['Test RMSE']:.2f}")

In [ ]:
# Medium Neural Network (Moderate)
print("\n13. Neural Network (Medium - Moderate)...")

nn_medium = PricePredictor(input_size, hidden_sizes=[128, 64, 32], dropout=0.3).to(device)
nn_medium_time = train_neural_network(nn_medium, train_loader, X_test_tensor, y_test_tensor, 
                                       epochs=50, lr=0.001)

y_pred_train_medium, y_pred_test_medium = evaluate_neural_network(
    nn_medium, X_train_tensor, X_test_tensor, y_train_tensor, y_test_tensor
)

nn_medium_metrics = {
    'Model': 'Neural Network (Medium)',
    'Complexity': '3 layers (128→64→32), 50 epochs',
    'Train Time (s)': round(nn_medium_time, 4),
    'Train R²': round(r2_score(y_train, y_pred_train_medium), 4),
    'Test R²': round(r2_score(y_test, y_pred_test_medium), 4),
    'Train RMSE': round(np.sqrt(mean_squared_error(y_train, y_pred_train_medium)), 2),
    'Test RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred_test_medium)), 2),
    'Train MAE': round(mean_absolute_error(y_train, y_pred_train_medium), 2),
    'Test MAE': round(mean_absolute_error(y_test, y_pred_test_medium), 2)
}
results.append(nn_medium_metrics)

print(f"  Training Time: {nn_medium_time:.4f}s ({device})")
print(f"  Test R²: {nn_medium_metrics['Test R²']:.4f}")
print(f"  Test RMSE: €{nn_medium_metrics['Test RMSE']:.2f}")

In [ ]:
# Large Neural Network (Slow but potentially more accurate)
print("\n14. Neural Network (Large - Deep)...")

nn_large = PricePredictor(input_size, hidden_sizes=[256, 128, 64, 32, 16], dropout=0.3).to(device)
nn_large_time = train_neural_network(nn_large, train_loader, X_test_tensor, y_test_tensor, 
                                      epochs=100, lr=0.0005)

y_pred_train_large, y_pred_test_large = evaluate_neural_network(
    nn_large, X_train_tensor, X_test_tensor, y_train_tensor, y_test_tensor
)

nn_large_metrics = {
    'Model': 'Neural Network (Large)',
    'Complexity': '5 layers (256→128→64→32→16), 100 epochs',
    'Train Time (s)': round(nn_large_time, 4),
    'Train R²': round(r2_score(y_train, y_pred_train_large), 4),
    'Test R²': round(r2_score(y_test, y_pred_test_large), 4),
    'Train RMSE': round(np.sqrt(mean_squared_error(y_train, y_pred_train_large)), 2),
    'Test RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred_test_large)), 2),
    'Train MAE': round(mean_absolute_error(y_train, y_pred_train_large), 2),
    'Test MAE': round(mean_absolute_error(y_test, y_pred_test_large), 2)
}
results.append(nn_large_metrics)

print(f"  Training Time: {nn_large_time:.4f}s ({device})")
print(f"  Test R²: {nn_large_metrics['Test R²']:.4f}")
print(f"  Test RMSE: €{nn_large_metrics['Test RMSE']:.2f}")

## 6. Model Comparison and Results

### 6.1 Results Summary Table

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Test R²', ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("FINAL RESULTS - ALL MODELS")
print("=" * 80)
print(results_df.to_string(index=False))

# Display as a formatted table
results_df

### 6.2 Visual Comparison: Performance vs Training Time

In [ ]:
# Performance vs Training Time scatter plot
fig = px.scatter(
    results_df, 
    x='Train Time (s)', 
    y='Test R²',
    text='Model',
    size='Test RMSE',
    color='Test R²',
    color_continuous_scale='RdYlGn',
    title='Model Performance vs Training Time Trade-off',
    labels={'Train Time (s)': 'Training Time (seconds)', 'Test R²': 'R² Score (Test)'},
    height=600
)

fig.update_traces(textposition='top center')
fig.update_layout(showlegend=False)
fig.show()

### 6.3 Model Performance Comparison

In [ ]:
# Bar chart of R² scores
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['R² Score Comparison', 'RMSE Comparison']
)

# R² scores
fig.add_trace(
    go.Bar(
        x=results_df['Model'], 
        y=results_df['Test R²'],
        marker_color='lightblue',
        name='R²'
    ),
    row=1, col=1
)

# RMSE
fig.add_trace(
    go.Bar(
        x=results_df['Model'], 
        y=results_df['Test RMSE'],
        marker_color='coral',
        name='RMSE'
    ),
    row=1, col=2
)

fig.update_xaxes(tickangle=-45)
fig.update_layout(height=500, showlegend=False)
fig.show()

### 6.4 Training Time Comparison

In [ ]:
# Training time comparison
results_time = results_df.sort_values('Train Time (s)')

fig = go.Figure()
fig.add_trace(go.Bar(
    y=results_time['Model'],
    x=results_time['Train Time (s)'],
    orientation='h',
    marker=dict(
        color=results_time['Train Time (s)'],
        colorscale='Viridis',
        showscale=True
    ),
    text=results_time['Train Time (s)'].round(2),
    textposition='auto'
))

fig.update_layout(
    title='Training Time Comparison Across Models',
    xaxis_title='Training Time (seconds)',
    yaxis_title='Model',
    height=600,
    showlegend=False
)
fig.show()

### 6.5 Best Model Analysis

In [ ]:
# Identify best models
best_r2 = results_df.loc[results_df['Test R²'].idxmax()]
best_rmse = results_df.loc[results_df['Test RMSE'].idxmin()]
fastest = results_df.loc[results_df['Train Time (s)'].idxmin()]

print("\n" + "=" * 80)
print("BEST MODELS BY CATEGORY")
print("=" * 80)

print(f"\n🏆 BEST R² SCORE: {best_r2['Model']}")
print(f"   Test R²: {best_r2['Test R²']:.4f}")
print(f"   Test RMSE: €{best_r2['Test RMSE']:.2f}")
print(f"   Training Time: {best_r2['Train Time (s)']:.2f}s")
print(f"   Complexity: {best_r2['Complexity']}")

print(f"\n🎯 LOWEST RMSE: {best_rmse['Model']}")
print(f"   Test RMSE: €{best_rmse['Test RMSE']:.2f}")
print(f"   Test R²: {best_rmse['Test R²']:.4f}")
print(f"   Training Time: {best_rmse['Train Time (s)']:.2f}s")

print(f"\n⚡ FASTEST MODEL: {fastest['Model']}")
print(f"   Training Time: {fastest['Train Time (s)']:.4f}s")
print(f"   Test R²: {fastest['Test R²']:.4f}")
print(f"   Test RMSE: €{fastest['Test RMSE']:.2f}")

# Best trade-off: R²/Time ratio
results_df['Efficiency'] = results_df['Test R²'] / results_df['Train Time (s)']
best_tradeoff = results_df.loc[results_df['Efficiency'].idxmax()]

print(f"\n⚖️ BEST PERFORMANCE/TIME TRADE-OFF: {best_tradeoff['Model']}")
print(f"   Efficiency Score: {best_tradeoff['Efficiency']:.4f}")
print(f"   Test R²: {best_tradeoff['Test R²']:.4f}")
print(f"   Training Time: {best_tradeoff['Train Time (s)']:.2f}s")

## 7. Conclusions

### Key Findings:

1. **Model Performance**:
   - Ensemble methods (Random Forest, XGBoost, LightGBM) generally outperformed linear models
   - Deep learning models showed competitive performance with proper tuning
   - GPU acceleration significantly reduced training time for boosting and neural network models

2. **Training Time Analysis**:
   - Linear models (Ridge, Lasso, ElasticNet) were fastest but had lower accuracy
   - GPU-accelerated models (XGBoost, LightGBM with CUDA) provided excellent performance with reasonable training times
   - Deep neural networks required more training time but offered flexibility in architecture

3. **Complexity vs Performance**:
   - Increasing model complexity didn't always lead to better generalization
   - Some simpler models (e.g., medium-sized Random Forest) achieved excellent results
   - Regularization in linear models helped prevent overfitting

4. **Feature Importance**:
   - Property characteristics (accommodates, bedrooms, bathrooms) were strong predictors
   - Location features (latitude, longitude, distance from center) significantly impacted price
   - Review scores and host characteristics contributed to model performance

### Recommendations:

- **For Production**: Use XGBoost or LightGBM with GPU for best balance of accuracy and speed
- **For Interpretability**: Ridge or Lasso regression with feature importance analysis
- **For Maximum Accuracy**: Ensemble multiple top-performing models
- **For Quick Prototyping**: Random Forest (medium size) for rapid development

### Future Work:

1. **Feature Engineering**: 
   - Extract more features from text descriptions (NLP)
   - Incorporate seasonal pricing patterns
   - Add neighborhood-level statistics

2. **Model Improvements**:
   - Hyperparameter optimization with Optuna
   - Model stacking and blending
   - Time-series analysis for price trends

3. **Deployment**:
   - Create REST API for price predictions
   - Build interactive dashboard
   - Implement model monitoring and retraining pipeline

## 8. Save Results and Models

In [ ]:
# Save results to CSV
results_df.to_csv('../outputs/model_comparison_results.csv', index=False)
print("Results saved to outputs/model_comparison_results.csv")

# Save feature names for reference
pd.DataFrame({'feature': feature_columns}).to_csv('../outputs/features.csv', index=False)
print("Feature list saved to outputs/features.csv")

print("\n✅ Analysis complete!")